# Cu–H Binary Phase Diagram

This notebook constructs a binary phase diagram for a fictitious Cu–H system using the `landau`
library together with ASE thermochemistry classes.  The concentration axis runs from pure Cu
($c = 0$) to pure H ($c = 1$).

The following phases are included:

| Phase | Type | Description |
|-------|------|-------------|
| `solid` | `SlowInterpolatingPhase` | Cu, CuH, and H endpoints interpolated |
| `liquid` | `SlowInterpolatingPhase` | Cu, Cu₃H, and CuH endpoints interpolated |
| `gas` | `IdealSolution` | Cu(g) + H₂(g) ideal mixture |
| CuH₂ | `LinePhase` | Intermetallic at $c = 2/3$ |

Thermochemical data are illustrative (toy HarmonicThermo and IdealGasThermo parameters).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ase import Atoms
from ase.build import molecule
from ase.thermochemistry import IdealGasThermo, HarmonicThermo

from landau.ase_phases import AsePhase
from landau.phases import IdealSolution, LinePhase, SlowInterpolatingPhase
from landau.calculate import calc_phase_diagram
from landau.plot import plot_phase_diagram, plot_mu_phase_diagram

## Solid Phase

The solid phase is modelled as an interpolated mixture of three `AsePhase` endpoints:
pure Cu (`c = 0`), CuH (`c = 0.5`), and H (`c = 1`).  Each endpoint uses `HarmonicThermo`
with three identical vibrational modes (Einstein model) and a DFT-like potential energy.

`SlowInterpolatingPhase` fits a smooth free-energy surface through these points at each temperature.
The `add_entropy=True` flag includes the ideal mixing entropy $-k_BT[c\ln c + (1-c)\ln(1-c)]$.

In [ ]:
sa  = AsePhase('Cu(s)',  0.0, HarmonicThermo([.1, .1, .1], potentialenergy=-2.00))
sab = AsePhase('CuH(s)', 0.5, HarmonicThermo([.1, .1, .1], potentialenergy=-2.25))
sb  = AsePhase('H(s)',   1.0, HarmonicThermo([.1, .1, .1], potentialenergy=-1.50))

s = SlowInterpolatingPhase('solid', [sa, sab, sb], add_entropy=True)

## Liquid Phase

The liquid phase uses the same structure but with softer phonon modes (lower frequencies → more
entropy) and slightly higher potential energies than the solid.  An additional intermediate
composition Cu₃H (`c = 0.25`) breaks the monotonic interpolation and creates a more complex
free-energy surface.

In [ ]:
la   = AsePhase('Cu(l)',   0.00, HarmonicThermo([.05]*3, potentialenergy=-1.80))
la3b = AsePhase('Cu3H(l)', 0.25, HarmonicThermo([.05]*3, potentialenergy=-2.15))
lab  = AsePhase('CuH(l)',  0.50, HarmonicThermo([.05]*3, potentialenergy=-2.20))

l = SlowInterpolatingPhase('liquid', [la, la3b, lab], add_entropy=True)

## Intermetallic and Gas Phases

**Intermetallic CuH₂** is a stoichiometric compound at $c = 2/3$.  It is represented as a
`LinePhase` with a fixed free energy (constant + small slope in temperature).

**Gas phase** is an ideal solution of monatomic Cu vapour (`c = 0`) and molecular H₂ (`c = 1`).
Each gas endpoint uses `IdealGasThermo` at a reference pressure of 10 kPa.
`IdealSolution` mixes the two endpoints using the ideal mixing entropy at the given pressure.

In [ ]:
im = LinePhase('CuH$_2$', 2/3, -2.25, .1e-5)

ga = AsePhase(
    'Cu(g)', 0.0,
    IdealGasThermo(
        [], 'monatomic',
        potentialenergy=2,
        atoms=Atoms('Cu'), symmetrynumber=0, spin=0,
    ),
    pressure=10000,
)
gb = AsePhase(
    'H(g)', 1.0,
    IdealGasThermo(
        [.2], 'linear',
        potentialenergy=-1.5,
        atoms=molecule('H2'), symmetrynumber=2, spin=0,
    ),
    pressure=10000,
)
g = IdealSolution('gas', ga, gb)

## Visualising the Free Energies

### Concentration dependence

The free energy of the solid and liquid interpolating phases as a function of composition at
$T = 100$ K.  Both phases show a non-trivial shape because of the intermediate endpoint (Cu₃H
in the liquid) and the ideal mixing entropy.

In [ ]:
cc = np.linspace(0, 1)[1:-1]

for p in (s, l):
    plt.plot(cc, p.free_energy(100, cc), label=p.name)
plt.xlabel('Composition $c$')
plt.ylabel('Free energy (eV/atom)')
plt.legend()
plt.title('Solid and liquid free energies at $T = 100$ K')

### Temperature dependence of the endpoint phases

The Helmholtz free energy of the Cu and H endpoint phases as a function of temperature,
comparing the solid and gas counterparts.

In [ ]:
t = np.linspace(10, 1000)

plt.subplot(121)
for p in (sa, ga):
    plt.plot(t, p.line_free_energy(t), label=p.name)
plt.xlabel('Temperature (K)')
plt.ylabel('Free energy (eV/atom)')
plt.legend()
plt.title('Cu endpoint')

plt.subplot(122)
for p in (sb, gb):
    plt.plot(t, p.line_free_energy(t), label=p.name)
plt.xlabel('Temperature (K)')
plt.legend()
plt.title('H endpoint')

plt.tight_layout()

### Semi-grand-canonical potential

The semi-grand-canonical potential $\phi(T, \Delta\mu) = F(T, c^*) - c^* \Delta\mu$ is the natural
thermodynamic potential for comparing phases at a fixed chemical potential difference
$\Delta\mu = \mu_H - \mu_{\rm Cu}$.  The stable phase at each $(T, \Delta\mu)$ is the one with the
lowest $\phi$.

Below we compare all phases at low ($T = 100$ K) and high ($T = 1000$ K) temperatures.

In [ ]:
m = np.linspace(-2, 2, 100)

for p in (sa, sb, ga, gb, im, s, l, g):
    plt.plot(m, p.semigrand_potential(100, m), label=p.name)
plt.xlabel(r'$\Delta\mu$ (eV)')
plt.ylabel(r'$\phi$ (eV/atom)')
plt.legend()
plt.title('Semi-grand-canonical potential at $T = 100$ K')

In [ ]:
for p in (sa, sb, ga, gb, im, s, l, g):
    plt.plot(m, p.semigrand_potential(1000, m), label=p.name)
plt.xlabel(r'$\Delta\mu$ (eV)')
plt.ylabel(r'$\phi$ (eV/atom)')
plt.legend()
plt.title('Semi-grand-canonical potential at $T = 1000$ K')

## Phase Diagram Calculation

`calc_phase_diagram` evaluates the semi-grand-canonical potential on a grid of $(T, \Delta\mu)$
points, determines the stable phase at each point, and refines phase boundaries to sub-grid precision.
The result is a `pandas.DataFrame` that can be visualised with `plot_mu_phase_diagram` (isothermal
slices in the $\Delta\mu$–$T$ plane) or `plot_phase_diagram` (phase regions in the $c$–$T$ plane).

In [ ]:
df = calc_phase_diagram(
    [s, l, g, im],
    Ts=np.linspace(10, 1750, 50),
    mu=np.linspace(-2.5, 3, 100),
    keep_unstable=True,
)

### Chemical potential – temperature diagram

Each region shows the stable phase in the $(\Delta\mu, T)$ plane.

In [ ]:
plot_mu_phase_diagram(df, poly_method='fasttsp')

### Composition – temperature diagram

The same information mapped to the physical composition axis $c$.  Phase boundaries are computed
from the lever rule at each temperature.  Two polygon methods are shown for comparison:
`fasttsp` (fast TSP-based hull) and `segments` (simple segment-based boundary).

In [ ]:
%%time
plot_phase_diagram(df, poly_method='fasttsp', tielines=True, min_c_width=.01)

In [ ]:
%%time
plot_phase_diagram(df, poly_method='segments', tielines=True, min_c_width=.01)